In [ ]:
"""
import os

os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.8"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
"""

In [ ]:
import os
import tensorflow as tf

# 1. تنظيف مسارات الكومبايلر لتناسب بيئة Linux (Lightning AI)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=2" # تفعيل أقصى تسريع لـ XLA
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" # إخفاء رسائل الـ Info المزعجة

# 2. إعدادات الـ GPU الحديثة (Lightning AI يستخدم عادة T4, L4, أو A10G)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # السماح بزيادة استهلاك الذاكرة تدريجياً لمنع الانهيار (OOM)
        tf.config.experimental.set_memory_growth(gpus[0], True)
        
        # --- التحديث السري للـ Transformers ---
        # تفعيل الـ Mixed Precision (FP16) 
        # هذا سيجعل الـ Transformer يتدرب أسرع بـ 2x ويستهلك نصف حجم الذاكرة (VRAM)!
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        
        print(f"GPU Initialized: {gpus[0].name}")
        print("Mixed Precision (FP16) Enabled for Faster Training!")
    except RuntimeError as e:
        print(e)
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")

###  Environment Setup and Library Imports


In [ ]:
# Standard System Libraries
import os
import sys
import gc
import time
import math
import random
import pickle
import glob
import datetime

# Data Manipulation & Visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from tqdm.autonotebook import tqdm

# Machine Learning & Evaluation Metrics
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, 
                             classification_report, confusion_matrix)

# Deep Learning (TensorFlow & Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (Dense, LSTM, Bidirectional, GRU, 
                                     Dropout, Input, LayerNormalization, 
                                     MultiHeadAttention, GlobalAveragePooling1D,
                                     Conv1D, MaxPooling1D)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision

# Configure random seeds for absolute reproducibility in research
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Print versions to document the research environment
print("--- Environment Details ---")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")



In [ ]:
""""
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def get_strategy():

    IS_TPU = False

    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        print("TPU detected")
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        IS_TPU = True

    except ValueError:

        gpus = tf.config.list_physical_devices('GPU')

        if len(gpus) > 1:
            print(f"{len(gpus)} GPUs detected")
            strategy = tf.distribute.MirroredStrategy()

        elif len(gpus) == 1:
            print("Single GPU detected")
            strategy = tf.distribute.get_strategy()

        else:
            print("No GPU detected, using CPU")
            strategy = tf.distribute.get_strategy()

    AUTO = tf.data.AUTOTUNE
    REPLICAS = strategy.num_replicas_in_sync

    print(f"Replicas in sync: {REPLICAS}")

    return strategy, REPLICAS, IS_TPU


seed_everything()

STRATEGY, N_REPLICAS, Ine_S_TPU = get_strategy()"""

In [ ]:
from pathlib import Path
import os

print("Current working directory:", Path().resolve())

DATA_DIR = Path("/mnt/Hub_1/Mix/Projects/Graduation-Project/data/Kaggl/asl-signs (2)/")

TRAIN_CSV = DATA_DIR / "train.csv"
LANDMARK_DIR = DATA_DIR / "train_landmark_files"
SIGN_MAP = DATA_DIR / "sign_to_prediction_index_map.json"

PROJECT_ROOT = Path("../")

EVAL_DIR = PROJECT_ROOT / "Evaluation_Plots"
MODEL_DIR = PROJECT_ROOT / "Saved_Models"
PRED_DIR = PROJECT_ROOT / "Predictions"
HIST_DIR = PROJECT_ROOT / "Training_Histories"

print("\nDataset paths check:")
print("DATA_DIR:", DATA_DIR)
print("Train CSV exists:", TRAIN_CSV.exists())
print("Landmark folder exists:", LANDMARK_DIR.exists())
print("Sign map exists:", SIGN_MAP.exists())

parquet_folders = list(LANDMARK_DIR.glob("*"))
print("Number of parquet folders:", len(parquet_folders))

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
display(train_df.head())
display(train_df.info())

In [ ]:
print("Number of samples in train.csv:", len(train_df))
print("Number of unique signs:", train_df["sign"].nunique())
print("Number of participants:", train_df["participant_id"].nunique())

### Spatial-Temporal Feature Engineering and Preprocessing
In this phase, we define the core preprocessing pipeline. Instead of feeding all 543 raw MediaPipe landmarks into the models, we isolate the most informative nodes (Lips, Eyes, Nose, and Hands) to reduce noise and computational complexity. 

Furthermore, we implement a custom Keras Layer (`Preprocess`) that performs the following operations directly within the TensorFlow graph:
1. **NaN Handling:** Computes safe means and standard deviations to normalize coordinates, replacing missing landmarks (NaNs) seamlessly.
2. **Normalization:** Centers the coordinates based on a reference point.
3. **Temporal Dynamics (Velocity & Acceleration):** Computes the first derivative (`dx`) and second derivative (`dx2`) of the coordinates across frames to capture motion speed and trajectory.
4. **Feature Fusion:** Concatenates positions, velocities, and accelerations into a robust feature vector (shape: `[Frames, Channels]`) optimized for sequential models.

In [ ]:
# Constants for data dimensions and padding
ROWS_PER_FRAME = 543
MAX_LEN = 384
CROP_LEN = MAX_LEN
NUM_CLASSES  = 250
PAD = -100.

# ---------------------------------------------------------------------------
# Feature Selection: Isolating Informative Landmarks (Lips, Nose, Eyes, Hands)
# ---------------------------------------------------------------------------
NOSE = [1, 2, 98, 327]
LNOSE = [98]
RNOSE = [327]

LIP = [ 
    0, 61, 185, 40, 39, 37, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
]

LLIP = [84, 181, 91, 146, 61, 185, 40, 39, 37, 87, 178, 88, 95, 78, 191, 80, 81, 82]
RLIP = [314, 405, 321, 375, 291, 409, 270, 269, 267, 317, 402, 318, 324, 308, 415, 310, 311, 312]

POSE = [500, 502, 504, 501, 503, 505, 512, 513]
LPOSE = [513, 505, 503, 501]
RPOSE = [512, 504, 502, 500]

REYE = [
    33, 7, 163, 144, 145, 153, 154, 155, 133,
    246, 161, 160, 159, 158, 157, 173,
]
LEYE = [
    263, 249, 390, 373, 374, 380, 381, 382, 362,
    466, 388, 387, 386, 385, 384, 398,
]

# MediaPipe Hand Landmarks indices
LHAND = np.arange(468, 489).tolist()
RHAND = np.arange(522, 543).tolist()

# Final concatenated feature list
POINT_LANDMARKS = LIP + LHAND + RHAND + NOSE + REYE + LEYE

NUM_NODES = len(POINT_LANDMARKS)
# Channels = (X, Y) * (Position, Velocity, Acceleration) = 2 * 3 = 6 per node
CHANNELS = 6 * NUM_NODES 

print(f"Total Selected Nodes: {NUM_NODES}")
print(f"Total Output Channels per frame: {CHANNELS}")

# ---------------------------------------------------------------------------
# Utility Functions for robust mathematical operations
# ---------------------------------------------------------------------------
def tf_nan_mean(x, axis=0, keepdims=False):
    """Computes the mean of a tensor ignoring NaN values."""
    sum_val = tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), x), axis=axis, keepdims=keepdims)
    count_val = tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), tf.ones_like(x)), axis=axis, keepdims=keepdims)
    return sum_val / count_val

def tf_nan_std(x, center=None, axis=0, keepdims=False):
    """Computes the standard deviation of a tensor ignoring NaN values."""
    if center is None:
        center = tf_nan_mean(x, axis=axis,  keepdims=True)
    d = x - center
    return tf.math.sqrt(tf_nan_mean(d * d, axis=axis, keepdims=keepdims))

# ---------------------------------------------------------------------------
# Custom Keras Layer for Spatial-Temporal Feature Engineering
# ---------------------------------------------------------------------------
class Preprocess(tf.keras.layers.Layer):
    """
    A custom TensorFlow layer that normalizes coordinates, handles NaNs, 
    and computes dynamic temporal features (velocity and acceleration).
    """
    def __init__(self, max_len=MAX_LEN, point_landmarks=POINT_LANDMARKS, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.point_landmarks = point_landmarks

    def call(self, inputs):
        if tf.rank(inputs) == 3:
            x = inputs[None, ...]
        else:
            x = inputs
        
        # Center normalization based on a reference point
        mean = tf_nan_mean(tf.gather(x, [17], axis=2), axis=[1, 2], keepdims=True)
        mean = tf.where(tf.math.is_nan(mean), tf.constant(0.5, x.dtype), mean)
        
        # Isolate selected landmarks
        x = tf.gather(x, self.point_landmarks, axis=2) # Shape: N, T, P, C
        std = tf_nan_std(x, center=mean, axis=[1, 2], keepdims=True)
        x = (x - mean) / std

        if self.max_len is not None:
            x = x[:, :self.max_len]
            
        length = tf.shape(x)[1]
        
        # Retain only X and Y coordinates (drop Z for classification efficiency)
        x = x[..., :2]

        # Calculate Velocity (First Derivative - dx)
        dx = tf.cond(
            tf.shape(x)[1] > 1,
            lambda: tf.pad(x[:, 1:] - x[:, :-1], [[0, 0], [0, 1], [0, 0], [0, 0]]),
            lambda: tf.zeros_like(x)
        )

        # Calculate Acceleration (Second Derivative - dx2)
        dx2 = tf.cond(
            tf.shape(x)[1] > 2,
            lambda: tf.pad(x[:, 2:] - x[:, :-2], [[0, 0], [0, 2], [0, 0], [0, 0]]),
            lambda: tf.zeros_like(x)
        )

        # Concatenate Position, Velocity, and Acceleration
        x = tf.concat([
            tf.reshape(x, (-1, length, 2 * len(self.point_landmarks))),
            tf.reshape(dx, (-1, length, 2 * len(self.point_landmarks))),
            tf.reshape(dx2, (-1, length, 2 * len(self.point_landmarks))),
        ], axis=-1)
        
        # Replace any remaining NaNs with zeros
        x = tf.where(tf.math.is_nan(x), tf.constant(0., x.dtype), x)
        
        return x

    def get_config(self):
        """Required for layer serialization and model saving."""
        config = super().get_config()
        config.update({
            "max_len": self.max_len,
            "point_landmarks": self.point_landmarks,
        })
        return config

### Phase 4: Data Augmentation and Parquet Pipeline Integration
This section adapts the standard TFRecord-based data loading pipeline to directly read from `.parquet` files using a Python generator wrapped in `tf.data.Dataset.from_generator`. 

It includes advanced spatial-temporal augmentations specifically designed for sign language recognition:
1. `flip_lr`: Simulates left-handed vs right-handed signers.
2. `resample`: Alters the speed of the sign dynamically.
3. `spatial_random_affine`: Applies rotation, scaling, and shear to simulate different camera angles.
4. `spatial_mask` & `temporal_mask`: Adds robustness by randomly obscuring parts of the frame or sequence.

Finally, the `get_parquet_dataset` function constructs an optimized, prefetching TensorFlow dataset ready for model training.

In [ ]:
# 1. Encode Sign Labels to Integers (0 to 249)
if 'label' not in train_df.columns:
    sign_list = sorted(train_df['sign'].unique())
    sign_to_label = {sign: label for label, sign in enumerate(sign_list)}
    label_to_sign = {label: sign for sign, label in sign_to_label.items()}
    train_df['label'] = train_df['sign'].map(sign_to_label)
    print(f"Encoded {len(sign_list)} unique signs.")

# Initialize the Preprocess layer defined in the previous cell
preprocess_layer = Preprocess(max_len=MAX_LEN, point_landmarks=POINT_LANDMARKS)

# ---------------------------------------------------------------------------
# Core Parquet Reader Function
# ---------------------------------------------------------------------------
def load_parquet_video(file_path):
    try:
        df = pd.read_parquet(file_path, columns=['x', 'y', 'z'], engine='fastparquet')
        coords = df.values.astype(np.float32)
        frames = len(coords) // ROWS_PER_FRAME
        return coords.reshape(frames, ROWS_PER_FRAME, 3)
    except Exception as e:
        return np.zeros((0, ROWS_PER_FRAME, 3), dtype=np.float32)

# ---------------------------------------------------------------------------
# Data Augmentation Functions (Corrected Tensor Dimensions)
# ---------------------------------------------------------------------------
def filter_nans_tf(x, ref_point=POINT_LANDMARKS):
    mask = tf.math.logical_not(tf.reduce_all(tf.math.is_nan(tf.gather(x, ref_point, axis=1)), axis=[-2, -1]))
    x = tf.boolean_mask(x, mask, axis=0)
    return x

def flip_lr(x):
    x_coord, y_coord, z_coord = tf.unstack(x, axis=-1)
    x_coord = 1 - x_coord
    new_x = tf.stack([x_coord, y_coord, z_coord], -1)
    new_x = tf.transpose(new_x, [1, 0, 2])
    
    lhand = tf.gather(new_x, LHAND, axis=0)
    rhand = tf.gather(new_x, RHAND, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LHAND)[..., None], rhand)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RHAND)[..., None], lhand)
    
    llip = tf.gather(new_x, LLIP, axis=0)
    rlip = tf.gather(new_x, RLIP, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LLIP)[..., None], rlip)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RLIP)[..., None], llip)
    
    lpose = tf.gather(new_x, LPOSE, axis=0)
    rpose = tf.gather(new_x, RPOSE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LPOSE)[..., None], rpose)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RPOSE)[..., None], lpose)
    
    leye = tf.gather(new_x, LEYE, axis=0)
    reye = tf.gather(new_x, REYE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LEYE)[..., None], reye)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(REYE)[..., None], leye)
    
    lnose = tf.gather(new_x, LNOSE, axis=0)
    rnose = tf.gather(new_x, RNOSE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LNOSE)[..., None], rnose)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RNOSE)[..., None], lnose)
    
    new_x = tf.transpose(new_x, [1, 0, 2])
    return new_x

def interp1d_(x, target_len, method='random'):
    target_len = tf.maximum(1, target_len)
    width = tf.shape(x)[1] # This represents the 543 landmarks
    size = [target_len, width]
    
    if method == 'random':
        rand_val = tf.random.uniform(())
        if rand_val < 0.33:
            x = tf.image.resize(x, size, 'bilinear')
        elif rand_val < 0.66:
            x = tf.image.resize(x, size, 'bicubic')
        else:
            x = tf.image.resize(x, size, 'nearest')
    else:
        x = tf.image.resize(x, size, method)
    return x

def resample(x, rate=(0.8, 1.2)):
    rate = tf.random.uniform((), rate[0], rate[1])
    length = tf.shape(x)[0]
    new_size = tf.cast(rate * tf.cast(length, tf.float32), tf.int32)
    new_x = interp1d_(x, new_size)
    return new_x

def spatial_random_affine(xyz, scale=(0.8, 1.2), shear=(-0.15, 0.15), shift=(-0.1, 0.1), degree=(-30, 30)):
    center = tf.constant([0.5, 0.5])
    if scale is not None:
        scale_val = tf.random.uniform((), *scale)
        xyz = scale_val * xyz

    if shear is not None:
        xy = xyz[..., :2]
        z = xyz[..., 2:]
        shear_x = shear_y = tf.random.uniform((), *shear)
        if tf.random.uniform(()) < 0.5:
            shear_x = 0.
        else:
            shear_y = 0.
        shear_mat = tf.identity([[1., shear_x], [shear_y, 1.]])
        xy = xy @ shear_mat
        center = center + [shear_y, shear_x]
        xyz = tf.concat([xy, z], axis=-1)

    if degree is not None:
        xy = xyz[..., :2]
        z = xyz[..., 2:]
        xy -= center
        degree_val = tf.random.uniform((), *degree)
        radian = degree_val / 180 * np.pi
        c = tf.math.cos(radian)
        s = tf.math.sin(radian)
        rotate_mat = tf.identity([[c, s], [-s, c]])
        xy = xy @ rotate_mat
        xy = xy + center
        xyz = tf.concat([xy, z], axis=-1)

    if shift is not None:
        shift_val = tf.random.uniform((), *shift)
        xyz = xyz + shift_val

    return xyz

def temporal_crop(x, length=MAX_LEN):
    l = tf.shape(x)[0]
    offset = tf.random.uniform((), 0, tf.clip_by_value(l - length, 1, length), dtype=tf.int32)
    x = x[offset:offset + length]
    return x

def temporal_mask(x, size=(0.2, 0.4), mask_value=float('nan')):
    l = tf.shape(x)[0]
    mask_size = tf.random.uniform((), *size)
    mask_size = tf.cast(tf.cast(l, tf.float32) * mask_size, tf.int32)
    mask_offset = tf.random.uniform((), 0, tf.clip_by_value(l - mask_size, 1, l), dtype=tf.int32)
    indices = tf.range(mask_offset, mask_offset + mask_size)[..., None]
    updates = tf.fill([mask_size, ROWS_PER_FRAME, 3], mask_value)
    x = tf.tensor_scatter_nd_update(x, indices, updates)
    return x

def spatial_mask(x, size=(0.2, 0.4), mask_value=float('nan')):
    mask_offset_y = tf.random.uniform(())
    mask_offset_x = tf.random.uniform(())
    mask_size = tf.random.uniform((), *size)
    mask_x = (mask_offset_x < x[..., 0]) & (x[..., 0] < mask_offset_x + mask_size)
    mask_y = (mask_offset_y < x[..., 1]) & (x[..., 1] < mask_offset_y + mask_size)
    mask = mask_x & mask_y
    x = tf.where(mask[..., None], mask_value, x)
    return x

def augment_fn(x, max_len=None):
    if tf.random.uniform(()) < 0.8:
        x = resample(x, (0.5, 1.5))
    if tf.random.uniform(()) < 0.5:
        x = flip_lr(x)
    if max_len is not None:
        x = temporal_crop(x, max_len)
    if tf.random.uniform(()) < 0.75:
        x = spatial_random_affine(x)
    if tf.random.uniform(()) < 0.5:
        x = temporal_mask(x)
    if tf.random.uniform(()) < 0.5:
        x = spatial_mask(x)
    return x

# ---------------------------------------------------------------------------
# TensorFlow Data Pipeline Implementation
# ---------------------------------------------------------------------------
def process_data(coord, label, augment=False, max_len=MAX_LEN):
    coord = filter_nans_tf(coord)
    if augment:
        coord = augment_fn(coord, max_len=max_len)
    coord = tf.ensure_shape(coord, (None, ROWS_PER_FRAME, 3))
    
    processed = preprocess_layer(coord)
    processed = tf.squeeze(processed, axis=0) 
    processed = tf.cast(processed, tf.float32)
    
    one_hot_label = tf.one_hot(label, NUM_CLASSES)
    return processed, one_hot_label

def get_parquet_dataset(df, data_dir=DATA_DIR, batch_size=64, max_len=MAX_LEN, augment=False, shuffle=False):
    def generator():
        sample_df = df.sample(frac=1).reset_index(drop=True) if shuffle else df
        for _, row in sample_df.iterrows():
            file_path = os.path.join(data_dir, str(row['path']).replace('\\', '/'))
            file_path = os.path.normpath(file_path)
            
            coords = load_parquet_video(file_path)
            label = int(row['label'])
            if coords.shape[0] > 0:
                yield coords, label

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, ROWS_PER_FRAME, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )

    ds = ds.map(lambda x, y: process_data(x, y, augment=augment, max_len=max_len), 
                num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.padded_batch(
        batch_size, 
        padding_values=(tf.cast(PAD, tf.float32), tf.cast(0.0, tf.float32)), 
        padded_shapes=([max_len, CHANNELS], [NUM_CLASSES]), 
        drop_remainder=True
    )
    
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# ---------------------------------------------------------------------------
# Pipeline Sanity Check
# ---------------------------------------------------------------------------
print("Testing the Parquet Pipeline...")
test_df_subset = train_df.head(10) 
test_ds = get_parquet_dataset(test_df_subset, batch_size=2, augment=True)

for batch_x, batch_y in test_ds.take(1):
    print(f"Batch X Shape: {batch_x.shape}")
    print(f"Batch Y Shape: {batch_y.shape}")
    break

###  Visualizing MediaPipe Landmarks 
Before feeding the data into complex neural networks, it is essential to visually verify the integrity of the spatial coordinates. The following code iterates through the parquet files, locates a sequence with valid hand landmarks (filtering out missing frames), and generates an interactive 2D animation of the hand skeletal connections over time. This confirms that the coordinate extraction and reshaping processes are correct.

In [ ]:
from IPython.display import HTML
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
import os
"""

# ---------------------------------------------------------
# MediaPipe Hand Connections
# Defines skeletal edges between the 21 hand landmarks
# ---------------------------------------------------------

HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(0,17),(17,18),(18,19),(19,20)
]


# ---------------------------------------------------------
# Utility: Filter frames that contain only NaN coordinates
# Some sequences contain missing frames that should be removed
# ---------------------------------------------------------

def filter_nans(frames):
    mask = ~np.isnan(frames).all(axis=(-2,-1))
    return frames[mask]


# ---------------------------------------------------------
# Locate a valid sequence in the dataset
# The sequence must contain a visible hand motion
# ---------------------------------------------------------

sample_frames = None

print("Searching for a valid sequence")

for row in train_df.itertuples():

    file_path = os.path.join(DATA_DIR, str(row.path).replace("\\","/"))
    file_path = os.path.normpath(file_path)

    coords = load_parquet_video(file_path)

    if coords.shape[0] == 0:
        continue

    lhand = coords[:,LHAND,:]

    valid_frames = filter_nans(lhand)

    if len(valid_frames) > 20:
        sample_frames = coords
        print("Sequence found:", row.sign)
        break


# ---------------------------------------------------------
# Core Animation Function
# Draws landmarks and skeleton edges frame-by-frame
# ---------------------------------------------------------

def animate_frames(frames, edges=None, idxs=None):

    frames = filter_nans(frames)

    fig, ax = plt.subplots(figsize=(6,6))

    def plot_frame(i):

        ax.clear()

        frame = np.nan_to_num(frames[i])

        x = frame[:,0]
        y = frame[:,1]

        ax.scatter(x,y,color="dodgerblue",s=40)

        if idxs is not None:
            for j in range(len(x)):
                ax.text(x[j],y[j],str(idxs[j]),fontsize=7)

        if edges is not None:
            for e in edges:
                ax.plot(
                    [x[e[0]],x[e[1]]],
                    [y[e[0]],y[e[1]]],
                    color="salmon",
                    linewidth=2
                )

        ax.invert_yaxis()

        ax.set_xticks([])
        ax.set_yticks([])

    anim = FuncAnimation(fig, plot_frame, frames=len(frames), interval=100)

    plt.close(fig)

    return HTML(anim.to_jshtml())


# ---------------------------------------------------------
# Save Animation Function
# Used to export animations for research paper figures
# ---------------------------------------------------------

def save_animation(frames, filename, edges=None, idxs=None):

    frames = filter_nans(frames)

    fig, ax = plt.subplots(figsize=(6,6))

    def plot_frame(i):

        ax.clear()

        frame = np.nan_to_num(frames[i])

        x = frame[:,0]
        y = frame[:,1]

        ax.scatter(x,y,color="dodgerblue",s=40)

        if idxs is not None:
            for j in range(len(x)):
                ax.text(x[j],y[j],str(idxs[j]),fontsize=7)

        if edges is not None:
            for e in edges:
                ax.plot([x[e[0]],x[e[1]]],[y[e[0]],y[e[1]]],color="salmon")

        ax.invert_yaxis()

        ax.set_xticks([])
        ax.set_yticks([])

    anim = FuncAnimation(fig, plot_frame, frames=len(frames), interval=100)

    save_path = os.path.join("Research Paper","Evaluation_Plots",filename)

    anim.save(save_path, writer="pillow", fps=10)

    plt.close(fig)

    print("Animation saved to:", save_path)


# ---------------------------------------------------------
# Display Left Hand Landmarks
# ---------------------------------------------------------

print("Left Hand Motion")

display(
    animate_frames(
        sample_frames[:,LHAND],
        edges=HAND_EDGES,
        idxs=list(range(len(LHAND)))
    )
)


# ---------------------------------------------------------
# Display Right Hand Landmarks
# ---------------------------------------------------------

print("Right Hand Motion")

display(
    animate_frames(
        sample_frames[:,RHAND],
        edges=HAND_EDGES,
        idxs=list(range(len(RHAND)))
    )
)


# ---------------------------------------------------------
# Display Face Landmarks
# ---------------------------------------------------------

print("Face Landmarks")

display(
    animate_frames(
        sample_frames[:,LIP + LEYE + REYE + NOSE],
        idxs=LIP + LEYE + REYE + NOSE
    )
)


# ---------------------------------------------------------
# Display All Selected Landmarks Used by the Model
# ---------------------------------------------------------

print("Full Landmark Representation")

display(
    animate_frames(
        sample_frames[:,POINT_LANDMARKS],
        idxs=POINT_LANDMARKS
    )
)


# ---------------------------------------------------------
# Example of Augmented Sequence Visualization
# ---------------------------------------------------------

print("Augmented Sequence Example")

augmented = augment_fn(sample_frames, max_len=MAX_LEN).numpy()

display(
    animate_frames(
        augmented[:,POINT_LANDMARKS],
        idxs=POINT_LANDMARKS
    )
)


# ---------------------------------------------------------
# Save animations for later use
# ---------------------------------------------------------

save_animation(sample_frames[:,RHAND],"right_hand.gif",edges=HAND_EDGES,idxs=list(range(len(RHAND))))
save_animation(sample_frames[:,POINT_LANDMARKS],"full_landmarks.gif",idxs=POINT_LANDMARKS)
"""

###  Final Data Shape and Tensor Inspection
Before defining the neural network architectures, we extract a single batch from our parquet dataset generator to inspect the exact tensor dimensions and the statistical properties of the engineered features. This confirms that the normalization, padding, and one-hot encoding have been applied correctly.

In [ ]:
import numpy as np

print("Fetching a single batch from the dataset...")
# We use the test_ds created in the previous cell
for batch_x, batch_y in test_ds.take(1):
    
    x_numpy = batch_x.numpy()
    y_numpy = batch_y.numpy()
    
    print("\n--- 1. Input Features (X) ---")
    print(f"Shape: {x_numpy.shape} -> (Batch, Frames, Channels)")
    print(f"Data Type: {x_numpy.dtype}")
    # Check normalization properties (should be centered around 0)
    print(f"Global Min Value: {np.min(x_numpy):.4f}")
    print(f"Global Max Value: {np.max(x_numpy):.4f}")
    print(f"Global Mean: {np.mean(x_numpy):.4f}")
    
    print("\n--- 2. Output Labels (Y) ---")
    print(f"Shape: {y_numpy.shape} -> (Batch, Num_Classes)")
    print(f"Data Type: {y_numpy.dtype}")
    
    # Show the active class for the first sample in the batch
    first_sample_label_index = np.argmax(y_numpy[0])
    # Assuming label_to_sign dictionary is available from previous cell
    sign_word = label_to_sign.get(first_sample_label_index, "Unknown")
    print(f"First Sample One-Hot Label Index: {first_sample_label_index}")
    print(f"Corresponding Sign Word: '{sign_word}'")
    
    print("\n Data is perfectly shaped and ready for modeling!")
    break

### Phase 2: Stratified Data Splitting and Dataset Creation
To ensure robust model evaluation and prevent data leakage, we split the dataset into Training (80%), Validation (10%), and Testing (10%) sets. 

Key considerations for this research pipeline:
1. **Stratification:** We stratify the split based on the target labels to maintain a consistent class distribution across all splits (crucial for the 250-class ISLR dataset).
2. **Reproducibility:** A fixed random seed ensures identical splits across different execution environments.
3. **Artifact Saving:** The splits are saved as CSV files. This guarantees that all subsequent baseline and advanced models, as well as the final Ensemble, are evaluated on the exact same unseen test instances.
4. **Optimized Pipelines:** We instantiate `tf.data` pipelines with `AUTOTUNE` prefetching. Augmentation and shuffling are strictly applied only to the training set.

In [ ]:

from sklearn.model_selection import train_test_split


print("Initiating Stratified Data Splitting...")

# 1. Create data directory if it doesn't exist (from your project structure)
os.makedirs("data", exist_ok=True)


train_df_split, temp_df = train_test_split(
    train_df, 
    test_size=0.20, 
    random_state=SEED, 
    stratify=train_df['label']
)

# Second split: Split the 40% Temporary equally into 20% Validation and 20% Test
val_df_split, test_df_split = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=SEED, 
    stratify=temp_df['label']
)

# 3. Save the splits to disk for absolute reproducibility during Ensemble
train_split_path = os.path.join("data", "train_split.csv")
val_split_path = os.path.join("data", "val_split.csv")
test_split_path = os.path.join("data", "test_split.csv")

train_df_split.to_csv(train_split_path, index=False)
val_df_split.to_csv(val_split_path, index=False)
test_df_split.to_csv(test_split_path, index=False)

print(f"Data Splitting Complete and Saved to 'data/' directory.")
print(f"Total Samples: {len(train_df)}")
print(f"--> Training Set:   {len(train_df_split)} samples ({len(train_df_split)/len(train_df)*100:.1f}%)")
print(f"--> Validation Set: {len(val_df_split)} samples ({len(val_df_split)/len(train_df)*100:.1f}%)")
print(f"--> Testing Set:    {len(test_df_split)} samples ({len(test_df_split)/len(train_df)*100:.1f}%)")

# 4. Create Highly Optimized tf.data.Datasets
print("\nConstructing TensorFlow Datasets...")

# Hyperparameters for training
BATCH_SIZE = 128 # Adjust this depending on your GPU RAM (e.g., 32 if OOM error occurs, 128 if plenty of VRAM)

# Train Dataset: Needs Augmentation and Shuffling
train_dataset = get_parquet_dataset(
    train_df_split, 
    data_dir=DATA_DIR, 
    batch_size=BATCH_SIZE, 
    max_len=MAX_LEN, 
    augment=True, 
    shuffle=True
)

# Validation Dataset: NO Augmentation, NO Shuffling (for accurate metric tracking)
val_dataset = get_parquet_dataset(
    val_df_split, 
    data_dir=DATA_DIR, 
    batch_size=BATCH_SIZE, 
    max_len=MAX_LEN, 
    augment=False, 
    shuffle=False
)

# Test Dataset: NO Augmentation, NO Shuffling (for final paper evaluation)
test_dataset = get_parquet_dataset(
    test_df_split, 
    data_dir=DATA_DIR, 
    batch_size=BATCH_SIZE, 
    max_len=MAX_LEN, 
    augment=False, 
    shuffle=False
)

print("\n--- TF Dataset Specifications ---")
print(f"Train Dataset: {train_dataset}")
print(f"Val Dataset:   {val_dataset}")
print(f"Test Dataset:  {test_dataset}")
print(" Data Pipelines are heavily optimized and ready for model consumption!")

### Dataset Summary and Pre-Training Report
Before initializing the training phase, we generate a comprehensive statistical report of the engineered dataset. This step verifies the integrity of the stratified split and the exact tensor dimensions. A textual summary is exported to the `results` directory, and a visual representation of the data distribution is saved to the `Evaluation_Plots` directory for inclusion in the research methodology section.

In [ ]:


# 1. Compile the Statistical Data
total_samples = len(train_df)
train_samples = len(train_df_split)
val_samples = len(val_df_split)
test_samples = len(test_df_split)

train_class_counts = train_df_split['label'].value_counts()
val_class_counts = val_df_split['label'].value_counts()
test_class_counts = test_df_split['label'].value_counts()

# 2. Generate the Textual Report
report_text = f"""
=========================================================
          WESSAL PROJECT: DATASET SUMMARY REPORT
=========================================================
1. GLOBAL DATASET METRICS
---------------------------------------------------------
Total Video Sequences Analyzed : {total_samples}
Total Unique Sign Classes      : {NUM_CLASSES}
Selected Landmarks per Frame   : {NUM_NODES} nodes
Engineered Feature Channels    : {CHANNELS} (X, Y, dx, dy, dx2, dy2)
Maximum Sequence Length        : {MAX_LEN} frames
Final Input Tensor Shape       : (Batch_Size, {MAX_LEN}, {CHANNELS})

2. STRATIFIED DATA SPLITTING (80/10/10)
---------------------------------------------------------
Training Set (80%)             : {train_samples} samples
Validation Set (10%)           : {val_samples} samples
Testing Set (10%)              : {test_samples} samples

3. CLASS BALANCE VERIFICATION (Samples per Class)
---------------------------------------------------------
[Training Set]   Max: {train_class_counts.max()} | Min: {train_class_counts.min()} | Mean: {train_class_counts.mean():.1f}
[Validation Set] Max: {val_class_counts.max()}  | Min: {val_class_counts.min()}  | Mean: {val_class_counts.mean():.1f}
[Testing Set]    Max: {test_class_counts.max()}  | Min: {test_class_counts.min()}  | Mean: {test_class_counts.mean():.1f}
=========================================================
"""

# Print to console
print(report_text)

# Save report to text file
report_path = os.path.join("../results", "Pre_Training_Dataset_Report.txt")
with open(report_path, "w") as text_file:
    text_file.write(report_text)
print(f"Report successfully saved to: {report_path}")

# 3. Generate Visual Report (Plots)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Pie Chart of the Split
labels = ['Training (60%)', 'Validation (20%)', 'Testing (20%)']
sizes = [train_samples, val_samples, test_samples]
colors = ['#4285F4', '#34A853', '#FBBC05']
explode = (0.05, 0, 0)  

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=False, startangle=90, textprops={'fontsize': 12})
ax1.axis('equal') 
ax1.set_title('Dataset Allocation (Stratified Split)', fontsize=14, fontweight='bold', pad=15)

# Subplot 2: Bar Chart showing Class Balance (Mean samples per class)
split_names = ['Train', 'Validation', 'Test']
mean_samples = [train_class_counts.mean(), val_class_counts.mean(), test_class_counts.mean()]

ax2.bar(split_names, mean_samples, color=['#4285F4', '#34A853', '#FBBC05'], width=0.5)
ax2.set_ylabel('Mean Sequences per Class', fontsize=12)
ax2.set_title('Average Class Representation per Split', fontsize=14, fontweight='bold', pad=15)

for i, v in enumerate(mean_samples):
    ax2.text(i, v + 2, f"{v:.1f}", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()

# Save the plot
plot_path = os.path.join("../Evaluation_Plots", "Data_Split_Distribution.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Visual distribution plot saved to: {plot_path}")

plt.show()

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

In [ ]:
import architecture_model.transformer_model as current_model_script

print("--- Initializing Model from External Script ---")

# Build the model using the imported script and global dimensions
model = current_model_script.get_transformer_model(input_shape=(MAX_LEN, CHANNELS), num_classes=NUM_CLASSES)
model_name = model.name

print(f" Successfully loaded architecture: {model_name}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import metrics, losses, optimizers

# -----------------------------------------------------------------------
# Cosine Decay with Linear Warmup
# Warmup: prevents large gradient updates when weights are random (first 7 epochs)
# -----------------------------------------------------------------------
steps_per_epoch = len(train_df_split) // BATCH_SIZE
total_steps     = steps_per_epoch * 100  # 100 epochs
warmup_steps    = steps_per_epoch * 7    # 7 epochs warmup

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.base_lr      = base_lr
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.min_lr       = min_lr

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step / self.warmup_steps)
        progress  = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress  = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
            1.0 + tf.cos(3.14159265 * progress)
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "base_lr":      self.base_lr,
            "warmup_steps": int(self.warmup_steps.numpy()),
            "total_steps":  int(self.total_steps.numpy()),
            "min_lr":       self.min_lr,
        }

schedule = WarmupCosineDecay(
    base_lr=1e-3,
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    min_lr=1e-6
)

advanced_optimizer = optimizers.AdamW(
    learning_rate=schedule,
    weight_decay=0.01,
    clipnorm=1.0
)

model.compile(
    optimizer=advanced_optimizer,
    loss=losses.CategoricalCrossentropy(
        from_logits=True,        
        label_smoothing=0.1
    ),
    metrics=[
        metrics.CategoricalAccuracy(name="accuracy"),
        metrics.TopKCategoricalAccuracy(k=5, name="top5_acc")
    ]
)

model.summary()

print(f"{model_name} compiled successfully with AdamW + Cosine Warmup Schedule and Label Smoothing.")
print(f"Steps per epoch : {steps_per_epoch}")
print(f"Warmup steps    : {warmup_steps}  (7 epochs)")
print(f"Total steps     : {total_steps} (100 epochs)")
print(f"LR range        : 1e-3 → 1e-6 (cosine)")


In [ ]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks
from tensorflow.keras.utils import plot_model



os.makedirs("../Evaluation_Plots", exist_ok=True)
os.makedirs("../logs", exist_ok=True)

# 1. Save Architecture Diagram
architecture_path = os.path.join("../Evaluation_Plots", f"{model_name}_architecture.png")
try:
    plot_model(model, to_file=architecture_path, show_shapes=True, show_layer_names=True, expand_nested=True, dpi=300)
    print(f"Architecture diagram saved: {architecture_path}")
except Exception as e:
    print("Note: Install 'pydot' and 'graphviz' to generate architecture plots.")

# 2. Log Model Parameters
params = model.count_params()
params_path = os.path.join("../Evaluation_Plots", f"{model_name}_params.txt")
with open(params_path, "w") as f:
    f.write(f"Total Parameters: {params:,}\n")
    f.write(f"Trainable Parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}\n")
print(f"Total Parameters: {params:,} (Saved to {params_path})")

# 3. Define Custom Callbacks
class TrainingPlot(callbacks.Callback):
    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        self.history_dict = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

    def on_epoch_end(self, epoch, logs=None):
        for key in self.history_dict.keys():
            if key in logs:
                self.history_dict[key].append(logs[key])

    def on_train_end(self, logs=None):
        # Accuracy Plot
        plt.figure(figsize=(10, 5))
        plt.plot(self.history_dict.get('accuracy', []))
        plt.plot(self.history_dict.get('val_accuracy', []))
        plt.title(f"{self.model_name} - Accuracy Curve", fontweight='bold')
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend(["Train", "Validation"])
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.savefig(os.path.join("../Evaluation_Plots", f"{self.model_name}_accuracy_curve.png"), dpi=300, bbox_inches='tight')
        plt.close()

        # Loss Plot
        plt.figure(figsize=(10, 5))
        plt.plot(self.history_dict.get('loss', []))
        plt.plot(self.history_dict.get('val_loss', []))
        plt.title(f"{self.model_name} - Loss Curve", fontweight='bold')
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend(["Train", "Validation"])
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.savefig(os.path.join("../Evaluation_Plots", f"{self.model_name}_loss_curve.png"), dpi=300, bbox_inches='tight')
        plt.close()

class BestMetricLogger(callbacks.Callback):
    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        self.best_val_acc = 0.0

    def on_epoch_end(self, epoch, logs=None):
        current_val_acc = logs.get('val_accuracy', 0.0)
        if current_val_acc > self.best_val_acc:
            self.best_val_acc = current_val_acc

    def on_train_end(self, logs=None):
        with open(os.path.join("../Evaluation_Plots", f"{self.model_name}_best_score.txt"), "w") as f:
            f.write(f"Best Validation Accuracy: {self.best_val_acc:.4f}\n")

# 4. Build Final Callbacks List
model_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join("../Saved_Models", f"{model_name}_best.h5"),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,    # تم تخفيضه إلى 15
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.CSVLogger(filename=os.path.join("../Training_Histories", f"{model_name}_log.csv"), append=False),
    TrainingPlot(model_name),
    BestMetricLogger(model_name),
    callbacks.TensorBoard(log_dir=os.path.join("../logs", model_name), histogram_freq=1)
]


In [ ]:
# Define the maximum number of epochs
# (EarlyStopping will likely stop it much earlier, usually around 30-50 epochs)
TRAINING_EPOCHS = 100

print(f"Maximum Epochs set to: {TRAINING_EPOCHS}")
print(f"Batch Size (handled by tf.data): {BATCH_SIZE}")

In [ ]:
# Sanity Check لـ شكل البيانات
for x, y in train_dataset.take(1):
    print(f"Input shape (X): {x.shape}")
    print(f"Labels shape (Y): {y.shape}")
    break

In [ ]:
print(f"Training for: {model_name}...")

# Start the training process
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=TRAINING_EPOCHS,
    callbacks=model_callbacks,
    verbose=1
)

print(f"\n Training Phase Completed for {model_name}!")

### 1. Training History
| Goal | Outputs |
| :--- | :--- |
| Trace the training process and detect overfitting/underfitting. | Accuracy and Loss curves over epochs. |

In [ ]:
print(f"--- Visualizing Training History for {model_name} ---")

# Load training history from the saved CSV log
log_path = os.path.join("../Training_Histories", f"{model_name}_log.csv")

if os.path.exists(log_path):
    history_df = pd.read_csv(log_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Accuracy Plot
    axes[0].plot(history_df['epoch'], history_df['accuracy'], label='Train Accuracy', color='#4285F4', linewidth=2)
    axes[0].plot(history_df['epoch'], history_df['val_accuracy'], label='Validation Accuracy', color='#34A853', linewidth=2)
    axes[0].set_title('Model Accuracy over Epochs', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Loss Plot
    axes[1].plot(history_df['epoch'], history_df['loss'], label='Train Loss', color='#EA4335', linewidth=2)
    axes[1].plot(history_df['epoch'], history_df['val_loss'], label='Validation Loss', color='#FBBC05', linewidth=2)
    axes[1].set_title('Model Loss over Epochs', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join("../Evaluation_Plots", f"{model_name}_Training_History.png"), dpi=300)
    plt.show()
else:
    print(f"Log file not found at {log_path}. Ensure the model has been trained.")

### 2. Comprehensive Evaluation
| Goal | Outputs |
| :--- | :--- |
| Comprehensive assessment of the model on entirely unseen data. | Overall Loss, Accuracy, and Top-5 Accuracy scores. |

In [ ]:
print(f"--- Executing Comprehensive Evaluation for {model_name} ---")

# Evaluate directly on the optimized test_dataset
eval_metrics = model.evaluate(test_dataset, verbose=1)

print("\n=========================================================")
print("             OVERALL TEST SET METRICS")
print("=========================================================")
print(f"Test Loss            : {eval_metrics[0]:.4f}")
print(f"Test Accuracy        : {eval_metrics[1]:.4f}  ({eval_metrics[1]*100:.2f}%)")
print(f"Test Top-5 Accuracy  : {eval_metrics[2]:.4f}  ({eval_metrics[2]*100:.2f}%)")
print("=========================================================")

### 3. Classification Report
| Goal | Outputs |
| :--- | :--- |
| Calculate foundational class-wise metrics. | CSV and TXT reports containing Precision, Recall, and F1-Score for all classes. |

In [ ]:
print("--- Generating Predictions and Classification Report ---")

y_true = []
y_pred_probs = []

# Extract true labels and compute predicted probabilities
for x_batch, y_batch in test_dataset:
    preds = model.predict(x_batch, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

# Generate Class Names mapping
target_names = [label_to_sign[i] for i in range(NUM_CLASSES)] if 'label_to_sign' in globals() else [str(i) for i in range(NUM_CLASSES)]

# Compute Classification Report
report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

# Export to CSV
csv_path = os.path.join("../results", f"{model_name}_Classification_Report.csv")
report_df.to_csv(csv_path)

# Print Summary
print(f"Full report saved to: {csv_path}")
print("\nGlobal Averages:")
print(f"-> Macro Avg F1-Score   : {report_dict['macro avg']['f1-score']:.4f}")
print(f"-> Weighted Avg F1-Score: {report_dict['weighted avg']['f1-score']:.4f}")

# Display top 5 rows of the dataframe
display(report_df.head())

### 4. Confusion Matrix
| Goal | Outputs |
| :--- | :--- |
| Understand specific model misclassifications and overlaps. | Absolute and Normalized Confusion Matrix high-resolution images. |

In [ ]:
print("--- Plotting Absolute and Normalized Confusion Matrices ---")

# 1. Absolute Confusion Matrix
cm_absolute = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(24, 20))
sns.heatmap(cm_absolute, cmap="Blues", cbar=True, xticklabels=False, yticklabels=False)
plt.title(f"{model_name} - Absolute Confusion Matrix", fontsize=22, pad=20)
plt.xlabel("Predicted Sign Class", fontsize=16)
plt.ylabel("True Sign Class", fontsize=16)
abs_path = os.path.join("../Evaluation_Plots", f"{model_name}_CM_Absolute.png")
plt.savefig(abs_path, dpi=300, bbox_inches='tight')
plt.close()

# 2. Normalized Confusion Matrix (Percentages)
cm_normalized = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(24, 20))
sns.heatmap(cm_normalized, cmap="rocket_r", cbar=True, xticklabels=False, yticklabels=False)
plt.title(f"{model_name} - Normalized Confusion Matrix (Recall per Class)", fontsize=22, pad=20)
plt.xlabel("Predicted Sign Class", fontsize=16)
plt.ylabel("True Sign Class", fontsize=16)
norm_path = os.path.join("../Evaluation_Plots", f"{model_name}_CM_Normalized.png")
plt.savefig(norm_path, dpi=300, bbox_inches='tight')
plt.close()

print(f" Absolute CM saved to: {abs_path}")
print(f" Normalized CM saved to: {norm_path}")

### 5. Class Performance
| Goal | Outputs |
| :--- | :--- |
| Analyze the individual performance of each sign class. | Horizontal bar charts highlighting the Top 20 Best and Top 20 Worst performing classes. |

In [ ]:
print("--- Analyzing Best and Worst Performing Classes ---")

# Extract only the 250 classes (remove 'accuracy', 'macro avg', 'weighted avg')
class_metrics = report_df.iloc[:-3].copy()
class_metrics = class_metrics.sort_values(by='f1-score', ascending=False)

best_20 = class_metrics.head(20)
worst_20 = class_metrics.tail(20).sort_values(by='f1-score', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Best 20 Plot
axes[0].barh(best_20.index[::-1], best_20['f1-score'][::-1], color='#34A853')
axes[0].set_title('Top 20 Performing Signs (Highest F1-Score)', fontsize=16, fontweight='bold')
axes[0].set_xlabel('F1-Score', fontsize=12)
axes[0].set_xlim(0, 1.05)
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# Worst 20 Plot
axes[1].barh(worst_20.index, worst_20['f1-score'], color='#EA4335')
axes[1].set_title('Top 20 Most Challenging Signs (Lowest F1-Score)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('F1-Score', fontsize=12)
axes[1].set_xlim(0, 1.05)
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
perf_path = os.path.join("../Evaluation_Plots", f"{model_name}_Class_Performance.png")
plt.savefig(perf_path, dpi=300)
plt.show()
print(f" Class performance charts saved to: {perf_path}")

### 6. Metrics Distribution
| Goal | Outputs |
| :--- | :--- |
| Visualize the statistical distribution of evaluation metrics across all classes. | Histograms detailing the density of Precision, Recall, and F1-Scores. |

In [ ]:
print("--- Plotting Metrics Distribution Across All Classes ---")

class_metrics = report_df.iloc[:-3]

plt.figure(figsize=(12, 6))
sns.kdeplot(class_metrics['precision'], fill=True, label='Precision', color='#4285F4', alpha=0.4)
sns.kdeplot(class_metrics['recall'], fill=True, label='Recall', color='#FBBC05', alpha=0.4)
sns.kdeplot(class_metrics['f1-score'], fill=True, label='F1-Score', color='#34A853', alpha=0.4)

plt.title("Distribution of Evaluation Metrics Across 250 Classes", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Metric Score", fontsize=14)
plt.ylabel("Density", fontsize=14)
plt.legend(fontsize=12)
plt.xlim(0, 1)
plt.grid(True, linestyle='--', alpha=0.5)

dist_path = os.path.join("../Evaluation_Plots", f"{model_name}_Metrics_Distribution.png")
plt.savefig(dist_path, dpi=300)
plt.show()
print(f"Distribution plot saved to: {dist_path}")

### 7. ROC Curves
| Goal | Outputs |
| :--- | :--- |
| Measure the discrimination ability of the classifier. | Receiver Operating Characteristic (ROC) curves and Area Under Curve (AUC) values for the macro-average and top challenging classes. |

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

print("--- Computing ROC Curves and AUC ---")

# Binarize labels for multi-class ROC calculation
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

# Compute micro-average ROC curve and ROC area
fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_pred_probs.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)

plt.figure(figsize=(10, 8))
plt.plot(fpr_micro, tpr_micro, label=f'Micro-average ROC (AUC = {roc_auc_micro:.3f})', 
         color='deeppink', linestyle=':', linewidth=4)

# Plot standard diagonal line
plt.plot([0, 1], [0, 1], 'k--', linewidth=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title(f'{model_name} - ROC Curve (Micro Average)', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

roc_path = os.path.join("../Evaluation_Plots", f"{model_name}_ROC_Curve.png")
plt.savefig(roc_path, dpi=300)
plt.show()
print(f" ROC Curve saved to: {roc_path}")

### 8. PR Curves
| Goal | Outputs |
| :--- | :--- |
| Evaluate model capability focusing on positive class prediction, especially useful for any inherent data imbalances. | Precision-Recall (PR) curves and Average Precision (AP) scores. |

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

print("--- Computing Precision-Recall Curves ---")

precision_micro, recall_micro, _ = precision_recall_curve(y_true_bin.ravel(), y_pred_probs.ravel())
ap_micro = average_precision_score(y_true_bin, y_pred_probs, average="micro")

plt.figure(figsize=(10, 8))
plt.plot(recall_micro, precision_micro, label=f'Micro-average PR (AP = {ap_micro:.3f})', 
         color='navy', linestyle='-', linewidth=3)

plt.xlabel('Recall', fontsize=14)
plt.ylabel('Precision', fontsize=14)
plt.title(f'{model_name} - Precision-Recall Curve (Micro Average)', fontsize=16, fontweight='bold')
plt.legend(loc="lower left", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

pr_path = os.path.join("../Evaluation_Plots", f"{model_name}_PR_Curve.png")
plt.savefig(pr_path, dpi=300)
plt.show()
print(f" PR Curve saved to: {pr_path}")

### 9. Error Analysis
| Goal | Outputs |
| :--- | :--- |
| Conduct a deep dive into common misclassifications to understand the root causes of confusion. | Identification and visualization of the most frequently confused sign pairs. |

In [ ]:
import collections

print("--- Conducting Deep Error Analysis (Most Confused Pairs) ---")

# Identify all misclassified instances
misclassified_indices = np.where(y_true != y_pred)[0]

# Create pairs of (True Label, Predicted Label)
error_pairs = [(target_names[y_true[i]], target_names[y_pred[i]]) for i in misclassified_indices]

# Count the frequency of each specific error pair
error_counts = collections.Counter(error_pairs)
top_errors = error_counts.most_common(10)

# Format for plotting
error_labels = [f"True: {pair[0][0]}\nPred: {pair[0][1]}" for pair in top_errors]
error_values = [pair[1] for pair in top_errors]

plt.figure(figsize=(14, 7))
sns.barplot(x=error_values, y=error_labels, palette="Reds_r")
plt.title(f"{model_name} - Top 10 Most Frequently Confused Sign Pairs", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Number of Misclassifications", fontsize=14)
plt.ylabel("Confusion Pair", fontsize=14)

# Add value labels
for index, value in enumerate(error_values):
    plt.text(value + 0.5, index, str(value), va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
error_path = os.path.join("../Evaluation_Plots", f"{model_name}_Top_Confusions.png")
plt.savefig(error_path, dpi=300)
plt.show()
print(f" Error analysis chart saved to: {error_path}")

###  Classification Report


In [ ]:
print("--- Extracting Predictions from Test Dataset ---")

y_true = []
y_pred_probs = []

# Iterate through the test dataset to gather true labels and model predictions
for x_batch, y_batch in test_dataset:
    preds = model.predict(x_batch, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Predictions extracted successfully.")

# Generate the classification report
print("\n--- Generating Classification Report ---")
# If label_to_sign mapping exists, use it; otherwise use numeric indices
target_names = [label_to_sign[i] for i in range(NUM_CLASSES)] if 'label_to_sign' in globals() else [str(i) for i in range(NUM_CLASSES)]

report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

# Save report to CSV
report_csv_path = os.path.join("../results", f"{model_name}_Classification_Report.csv")
report_df.to_csv(report_csv_path)

print(f"Classification report saved to: {report_csv_path}")

# Display macro and weighted averages
print("\nGlobal Metrics:")
print(f"Macro Avg F1-Score   : {report_dict['macro avg']['f1-score']:.4f}")
print(f"Weighted Avg F1-Score: {report_dict['weighted avg']['f1-score']:.4f}")

### Error Analysis (Lowest Performing Classes)

In [ ]:


# Extract F1-scores for individual classes (excluding macro/weighted avgs)
class_metrics = report_df.iloc[:-3]
class_metrics = class_metrics.sort_values(by='f1-score', ascending=True)

# Select the bottom 20 performing classes
bottom_20 = class_metrics.head(20)

plt.figure(figsize=(12, 8))
plt.barh(bottom_20.index, bottom_20['f1-score'], color='#E53935')

plt.title(f"{model_name} - Top 20 Most Challenging Signs (Lowest F1-Score)", fontsize=16, fontweight='bold')
plt.xlabel("F1-Score", fontsize=12)
plt.ylabel("Sign Class", fontsize=12)
plt.xlim(0, 1.0)
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Add exact numbers on the bars
for index, value in enumerate(bottom_20['f1-score']):
    plt.text(value + 0.01, index, f"{value:.2f}", va='center', fontsize=10)

plt.tight_layout()

# Save the plot
worst_classes_path = os.path.join("../Evaluation_Plots", f"{model_name}_Worst_Classes.png")
plt.savefig(worst_classes_path, dpi=300)
plt.close()

print(f"Error analysis plot saved to: {worst_classes_path}")

In [ ]:
import subprocess
import sys

# Run without --quiet to see the actual error
result = subprocess.run([
    "pip", "install",
    "protobuf==3.19.6",
    "onnx==1.13.0",
    "tf2onnx==1.14.0",
    "--force-reinstall"
], capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

In [ ]:
import os
import tensorflow as tf

def export_universal_model(model, model_name, saved_dir="../Saved_Models", max_len=384, channels=708):
    print(f"\n=========================================================")
    print(f"   STARTING UNIVERSAL EXPORT FOR: {model_name}")
    print(f"=========================================================")

    os.makedirs(saved_dir, exist_ok=True)

    # 1. Export Standard Keras Model
    keras_path = os.path.join(saved_dir, f"{model_name}_final.keras")
    try:
        model.save(keras_path)
        print(f"[1/2]  Standard Keras model saved: {keras_path}")
    except Exception as e:
        print(f"[1/2]  Failed to save Keras model: {e}")

    # 2. Export TFLite Model
    tflite_path = os.path.join(saved_dir, f"{model_name}.tflite")
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter._experimental_lower_tensor_list_ops = False
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
        print(f"[2/2]  TFLite model saved (Mobile Ready): {tflite_path}")
    except Exception as e:
        print(f"[2/2]  Failed to save TFLite model: {e}")

    # 3. ONNX Export — requires kernel restart, run separately after training
    print(f"[SKIP] ONNX export skipped — run export_onnx_after_restart.py after kernel restart")
    onnx_script_path = os.path.join(saved_dir, "export_onnx_after_restart.py")
    with open(onnx_script_path, "w") as f:
        f.write(f"""import tensorflow as tf
import tf2onnx

model = tf.keras.models.load_model(r"{keras_path}")
input_signature = [tf.TensorSpec([None, {max_len}, {channels}], tf.float32, name='input_features')]
tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13, output_path=r"{os.path.join(saved_dir, f'{model_name}.onnx')}")
print("ONNX export done.")
""")
    print(f"[INFO] ONNX script saved to: {onnx_script_path}")
    print(f"       Run it after kernel restart with: python {onnx_script_path}")

    print(f"=========================================================")
    print(f"   EXPORT PIPELINE COMPLETED FOR: {model_name}")
    print(f"=========================================================\n")

export_universal_model(model, model_name=model.name, max_len=MAX_LEN, channels=CHANNELS)